#Read data from Bronze table for crm_cust_info_raw table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
df = spark.table("workspace.bronze.crm_cust_info_raw")

# Transform the data

## Trimming

In [0]:
RENAME_MAP = {
    "cst_id" : "customer_id",
    "cst_key" : "customer_key",
    "cst_firstname" : "first_name",
    "cst_lastname" : "last_name",
    "cst_marital_status" : "marital_status",
    "cst_gndr" : "gender",
    "cst_create_date" : "created_date" 
}

In [0]:
# change col names
# trim additional space at start and end of string

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization the abbrevations

In [0]:
df = (
    df
    .withColumn("cst_marital_status", 
                F.when(F.upper(F.col("cst_marital_status")) =="S", "Single")
                .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
                .otherwise("n/a")
    )
    .withColumn("cst_gndr", 
                F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
                .when(F.upper(F.col("cst_gndr")) == "F", "Female")
                .otherwise("n/a")
    )
)

## Renaming The Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

#Write data to Silver table crm_cust_info_raw table

In [0]:
(
df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.silver.crm_cust_info")
)

# Reading from silver cust_info to see if data got written properly

In [0]:
df = spark.table("workspace.silver.crm_cust_info")
df.display()